# Identificação de Espécies de Pássaros — POC com Visão Computacional Clássica

**Dataset:** Backyard Feeder Birds (NABirds Subset)  
**Objetivo:** Construir uma prova de conceito para classificar espécies de pássaros usando técnicas clássicas de visão computacional, sem deep learning.

## Pipeline
1. Setup do ambiente e carregamento do dataset
2. Exploração do dataset
3. Pré-processamento das imagens
4. Segmentação clássica com múltiplas estratégias
5. Extração de features clássicas sobre a região segmentada
6. Treinamento de classificador simples
7. Avaliação


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.setup import setup

DATA_DIR = setup()
print("Dataset root:", DATA_DIR)


In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

from pathlib import Path
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC

from src.utils import create_run_dir, save_fig, save_metrics

%matplotlib inline
matplotlib.rcParams["figure.figsize"] = (14, 6)

RUN_DIR = create_run_dir()


## Descoberta automática da estrutura do dataset

Esta célula tenta localizar pastas que contenham imagens e assumir que o nome da pasta representa a classe.

In [ ]:
DATASET = Path(DATA_DIR)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def find_class_folders(root: Path):
    class_dirs = []
    for subdir in root.rglob("*"):
        if subdir.is_dir():
            try:
                files = [p for p in subdir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
                if files:
                    class_dirs.append(subdir)
            except Exception:
                pass
    return sorted(set(class_dirs))

class_dirs = find_class_folders(DATASET)
print(f"Found {len(class_dirs)} candidate class folders")
for d in class_dirs[:20]:
    print(d)


In [ ]:
rows = []
for class_dir in class_dirs:
    class_name = class_dir.name
    for img_path in class_dir.iterdir():
        if img_path.suffix.lower() in IMAGE_EXTS:
            rows.append({
                "class_name": class_name,
                "image_path": str(img_path)
            })

df = pd.DataFrame(rows)
print(df.head())
print()
print("Total images:", len(df))
print("Total classes:", df["class_name"].nunique())


## Distribuição das classes

In [ ]:
class_counts = df["class_name"].value_counts()

plt.figure(figsize=(14, 5))
class_counts.plot(kind="bar")
plt.title("Número de imagens por classe")
plt.ylabel("Quantidade")
plt.xlabel("Classe")
plt.xticks(rotation=90)
plt.tight_layout()
save_fig("01_class_distribution", RUN_DIR)
plt.show()


## Amostras do dataset

In [ ]:
def load_rgb(path: str):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

sample_df = df.groupby("class_name").head(2).reset_index(drop=True)
n = min(len(sample_df), 12)
sample_df = sample_df.iloc[:n]

rows_plot = int(np.ceil(n / 2))
fig, axes = plt.subplots(rows_plot, 2, figsize=(10, 5 * rows_plot))
axes = np.array(axes).reshape(-1)

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    img = load_rgb(row["image_path"])
    ax.imshow(img)
    ax.set_title(row["class_name"])
    ax.axis("off")

for ax in axes[len(sample_df):]:
    ax.axis("off")

plt.tight_layout()
save_fig("02_samples", RUN_DIR)
plt.show()


## Pré-processamento

In [ ]:
IMG_SIZE = (160, 160)

def preprocess_image(path: str):
    img_bgr = cv2.imread(path)
    img_bgr = cv2.resize(img_bgr, IMG_SIZE)

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_blur = cv2.GaussianBlur(img_gray, (5, 5), 0)

    return img_rgb, img_hsv, img_gray, img_blur


## Segmentação clássica com múltiplas estratégias

In [ ]:
def largest_component(mask):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    if num_labels <= 1:
        return mask

    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_idx = 1 + np.argmax(areas)

    out = np.zeros_like(mask)
    out[labels == largest_idx] = 255
    return out


def clean_mask(mask, k_open=3, k_close=5):
    kernel_open = np.ones((k_open, k_open), np.uint8)
    kernel_close = np.ones((k_close, k_close), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)
    mask = largest_component(mask)
    return mask


def segment_otsu(img_blur):
    _, mask = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return clean_mask(mask)


def segment_otsu_inverse(img_blur):
    _, mask = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return clean_mask(mask)


def segment_adaptive(img_blur):
    mask = cv2.adaptiveThreshold(
        img_blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        21,
        4
    )
    return clean_mask(mask)


def segment_edges(img_blur):
    edges = cv2.Canny(img_blur, 60, 140)
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.dilate(edges, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    return clean_mask(mask)


def segment_hsv_saturation(img_hsv):
    mask = cv2.inRange(img_hsv, (0, 25, 20), (179, 255, 255))
    return clean_mask(mask)


def segment_grabcut_rect(img_rgb):
    img = img_rgb.copy()
    mask = np.zeros(img.shape[:2], np.uint8)

    h, w = img.shape[:2]
    rect = (
        int(w * 0.10),
        int(h * 0.10),
        int(w * 0.80),
        int(h * 0.80)
    )

    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)

    try:
        cv2.grabCut(img, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
        out = np.where(
            (mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD),
            255,
            0
        ).astype("uint8")
        return clean_mask(out)
    except Exception:
        return np.zeros(img.shape[:2], dtype=np.uint8)


def mask_score(mask):
    area = np.count_nonzero(mask)
    h, w = mask.shape
    area_ratio = area / (h * w + 1e-8)

    if area == 0:
        return -1e9

    score = 0.0
    if 0.03 <= area_ratio <= 0.65:
        score += 2.0
    else:
        score -= 2.0

    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels > 1:
        largest_area = stats[1:, cv2.CC_STAT_AREA].max()
        score += largest_area / (area + 1e-8)

    return score


def build_segmentation_candidates(img_rgb, img_hsv, img_gray, img_blur):
    candidates = {
        "otsu": segment_otsu(img_blur),
        "otsu_inv": segment_otsu_inverse(img_blur),
        "adaptive": segment_adaptive(img_blur),
        "edges": segment_edges(img_blur),
        "hsv_sat": segment_hsv_saturation(img_hsv),
        "grabcut": segment_grabcut_rect(img_rgb)
    }
    return candidates


def choose_best_mask(candidates):
    best_name = None
    best_mask = None
    best_score = -1e18

    for name, mask in candidates.items():
        score = mask_score(mask)
        if score > best_score:
            best_score = score
            best_name = name
            best_mask = mask

    return best_name, best_mask, best_score


def apply_mask(img_rgb, mask):
    out = img_rgb.copy()
    out[mask == 0] = 0
    return out


def segment_bird(path: str):
    img_rgb, img_hsv, img_gray, img_blur = preprocess_image(path)
    candidates = build_segmentation_candidates(img_rgb, img_hsv, img_gray, img_blur)
    best_name, best_mask, best_score = choose_best_mask(candidates)
    segmented = apply_mask(img_rgb, best_mask)

    return {
        "rgb": img_rgb,
        "hsv": img_hsv,
        "gray": img_gray,
        "blur": img_blur,
        "candidates": candidates,
        "best_name": best_name,
        "best_mask": best_mask,
        "best_score": best_score,
        "segmented": segmented
    }


## Visualização das segmentações

In [ ]:
def show_segmentation_gallery(path: str):
    result = segment_bird(path)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    axes[0].imshow(result["rgb"])
    axes[0].set_title("RGB")
    axes[0].axis("off")

    axes[1].imshow(result["gray"], cmap="gray")
    axes[1].set_title("Gray")
    axes[1].axis("off")

    names = list(result["candidates"].keys())
    for i, name in enumerate(names[:4], start=2):
        axes[i].imshow(result["candidates"][name], cmap="gray")
        axes[i].set_title(name)
        axes[i].axis("off")

    for ax_idx, name in zip([6, 7], names[4:6]):
        axes[ax_idx].imshow(result["candidates"][name], cmap="gray")
        axes[ax_idx].set_title(name)
        axes[ax_idx].axis("off")

    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(result["rgb"])
    ax[0].set_title("Original")
    ax[0].axis("off")

    ax[1].imshow(result["best_mask"], cmap="gray")
    ax[1].set_title(f"Best mask: {result['best_name']}")
    ax[1].axis("off")

    ax[2].imshow(result["segmented"])
    ax[2].set_title("Segmented bird")
    ax[2].axis("off")

    plt.tight_layout()
    plt.show()

    return result


In [ ]:
row = df.sample(1, random_state=42).iloc[0]
print(row["class_name"])
seg_result = show_segmentation_gallery(row["image_path"])


## Exemplos de segmentação salvos

In [ ]:
sample_paths = df.sample(6, random_state=123).reset_index(drop=True)

fig, axes = plt.subplots(6, 3, figsize=(12, 24))

for i, (_, row) in enumerate(sample_paths.iterrows()):
    res = segment_bird(row["image_path"])

    axes[i, 0].imshow(res["rgb"])
    axes[i, 0].set_title(f"Original\n{row['class_name']}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(res["best_mask"], cmap="gray")
    axes[i, 1].set_title(f"Mask ({res['best_name']})")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(res["segmented"])
    axes[i, 2].set_title("Segmented")
    axes[i, 2].axis("off")

plt.tight_layout()
save_fig("03_segmentation_examples", RUN_DIR)
plt.show()


## Extração de features clássicas sobre a região segmentada

In [ ]:
def color_features_masked(img_hsv, mask):
    feats = []

    for ch, bins, rng in [(0, 16, [0, 180]), (1, 16, [0, 256]), (2, 16, [0, 256])]:
        hist = cv2.calcHist([img_hsv], [ch], mask, [bins], rng).flatten()
        hist = hist.astype(np.float32)
        hist /= (hist.sum() + 1e-8)
        feats.append(hist)

    return np.concatenate(feats).astype(np.float32)


def lbp_image(gray):
    h, w = gray.shape
    lbp = np.zeros((h - 2, w - 2), dtype=np.uint8)

    center = gray[1:-1, 1:-1]
    offsets = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, 1), (1, 1), (1, 0),
        (1, -1), (0, -1)
    ]

    for i, (dy, dx) in enumerate(offsets):
        neighbor = gray[1 + dy:h - 1 + dy, 1 + dx:w - 1 + dx]
        lbp |= ((neighbor >= center) << i).astype(np.uint8)

    return lbp


def texture_features_masked(img_gray, mask):
    lbp = lbp_image(img_gray)
    mask_inner = mask[1:-1, 1:-1]
    vals = lbp[mask_inner > 0]

    if len(vals) == 0:
        return np.zeros(32, dtype=np.float32)

    hist, _ = np.histogram(vals, bins=32, range=(0, 256))
    hist = hist.astype(np.float32)
    hist /= (hist.sum() + 1e-8)
    return hist


def shape_features_from_mask(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return np.zeros(12, dtype=np.float32)

    c = max(contours, key=cv2.contourArea)

    area = cv2.contourArea(c)
    perimeter = cv2.arcLength(c, True)

    x, y, w, h = cv2.boundingRect(c)
    aspect_ratio = w / (h + 1e-8)
    extent = area / (w * h + 1e-8)

    hull = cv2.convexHull(c)
    hull_area = cv2.contourArea(hull)
    solidity = area / (hull_area + 1e-8)

    equivalent_diameter = np.sqrt(4 * area / np.pi) if area > 0 else 0.0

    moments = cv2.moments(c)
    hu = cv2.HuMoments(moments).flatten()
    hu = np.sign(hu) * np.log1p(np.abs(hu))

    return np.concatenate([
        np.array([
            area,
            perimeter,
            aspect_ratio,
            extent,
            solidity,
            equivalent_diameter
        ], dtype=np.float32),
        hu.astype(np.float32)
    ])


def edge_features_masked(img_gray, mask):
    edges = cv2.Canny(img_gray, 60, 140)
    edge_ratio = np.count_nonzero((edges > 0) & (mask > 0)) / (np.count_nonzero(mask) + 1e-8)

    gx = cv2.Sobel(img_gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)

    vals = mag[mask > 0]
    if len(vals) == 0:
        return np.zeros(9, dtype=np.float32)

    hist, _ = np.histogram(vals, bins=8, range=(0, np.max(vals) + 1e-8))
    hist = hist.astype(np.float32)
    hist /= (hist.sum() + 1e-8)

    return np.concatenate([hist, np.array([edge_ratio], dtype=np.float32)])


def extract_features(path: str):
    result = segment_bird(path)

    img_hsv = result["hsv"]
    img_gray = result["gray"]
    mask = result["best_mask"]

    f_color = color_features_masked(img_hsv, mask)
    f_texture = texture_features_masked(img_gray, mask)
    f_shape = shape_features_from_mask(mask)
    f_edge = edge_features_masked(img_gray, mask)

    feat = np.concatenate([f_color, f_texture, f_shape, f_edge]).astype(np.float32)
    return feat


## Seleção da POC

Como o dataset é grande, vamos trabalhar com as classes mais frequentes e limitar o número de imagens por classe.

In [ ]:
TOP_N_CLASSES = 25
MAX_PER_CLASS = 60

top_classes = df["class_name"].value_counts().head(TOP_N_CLASSES).index

df_small = (
    df[df["class_name"].isin(top_classes)]
      .groupby("class_name", group_keys=False)
      .head(MAX_PER_CLASS)
      .reset_index(drop=True)
)

print("Images in POC:", len(df_small))
print("Classes in POC:", df_small["class_name"].nunique())


## Montagem do dataset de features

In [ ]:
X = []
y = []

for _, row in tqdm(df_small.iterrows(), total=len(df_small)):
    try:
        feat = extract_features(row["image_path"])
        X.append(feat)
        y.append(row["class_name"])
    except Exception as e:
        print("Skipping:", row["image_path"], e)

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


## Classificação com SVM

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=5, gamma="scale"))
])

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))


## Matriz de confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
save_fig("04_confusion_matrix", RUN_DIR)
plt.show()


## Salvando métricas

In [ ]:
metrics = {
    "dataset_slug": "jakemccaig/backyard-feeder-birds-nabirds-subset",
    "num_images": int(len(X)),
    "num_classes": int(len(le.classes_)),
    "feature_dim": int(X.shape[1]),
    "classifier": "SVM",
    "accuracy": float(acc)
}

save_metrics(metrics, RUN_DIR)


## Próximos passos

Possíveis extensões da POC:
- comparar SVM vs kNN
- comparar features com e sem segmentação
- testar diferentes estratégias de escolha de máscara
- avaliar impacto isolado de cor, textura, forma e bordas
